# Chunking Strategies — Hands-On

**LLM Engineering Track · Domain 1 (Data Representation) · Roadmap Weeks 13–14**

Companion notebook to the literature note `02 Literature Notes/LLM Engineering/Chunking Strategies`
and the deck `07 Resources Library/LLM Engineering/Slides/Lesson_01_Chunking_Strategies.pptx`.

You will implement and compare every strategy on real text:
1. Recursive + token-aware (the production default)
2. Structure-aware (Markdown / code)
3. Semantic chunking **from scratch** (the math, no framework)
4. Contextual retrieval (Anthropic pattern)
5. A **recall@k / MRR evaluation harness** to pick a config with evidence

**Sources:** Anthropic *Contextual Retrieval* (2024); Pinecone *Chunking Strategies* (2025);
Jina AI *Late Chunking* (2024); LangChain text-splitter docs.


## 0. Setup

Install dependencies. OpenAI calls are optional — the notebook degrades gracefully to a local hashing embedder so it runs with **no API key**.

In [ ]:
%pip install -q langchain-text-splitters tiktoken numpy openai
import os, re, numpy as np
np.set_printoptions(precision=3, suppress=True)
print("ready")

We use a small sample document with clear topic shifts so the strategies are easy to compare.

In [ ]:
DOC = """Acme Corp 2023 Annual Report — Risk Factors.
Our supply chain faced significant disruption in early 2023. A new supplier
contract signed in March restructured our logistics network. This cut costs by
40% versus the prior year. Management considers supplier concentration a key risk.

Financial Performance. Revenue grew 12% year over year to $4.2B. Gross margin
expanded 300 basis points driven by the logistics savings above. Operating income
reached $610M. The board approved a dividend increase of 8%.

Research and Development. We invested $520M in R&D, focused on battery chemistry.
Two new patents were granted for solid-state cells. The R&D team expanded to 900
engineers across three sites. Time-to-market for the next platform is 2025."""
print(DOC[:200])

## 1. Tokens, not characters

Chunk size must be measured in **tokens** — the unit the embedder and LLM consume,
and the unit you pay for. Character sizing silently truncates chunks that exceed
the embedding model's context window (Pinecone).

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # gpt-4o / text-embedding-3 family
tokens = enc.encode(DOC)
print(f"{len(DOC)} characters  =  {len(tokens)} tokens  "
      f"(~{len(DOC)/len(tokens):.1f} chars/token)")

### The chunk-count formula

For a document of `T` tokens, size `c`, overlap `o`, stride `s = c - o`:

$$N = \lceil (T - o) / (c - o) \rceil$$

More overlap ⇒ smaller stride ⇒ more chunks ⇒ more storage and embedding cost.

In [ ]:
import math
def n_chunks(T, c, o):
    return math.ceil((T - o) / (c - o))

T = len(tokens)
for c, o in [(500, 0), (500, 75), (300, 45), (800, 120)]:
    print(f"size={c:4d} overlap={o:3d} -> {n_chunks(T, c, o)} chunks "
          f"(stride {c-o})")

## 2. Recursive + token-aware (the default)

Split on the largest natural separator first (`\n\n`), recurse to smaller ones
only when a piece is still too big. Size measured with the real tokenizer, and
metadata attached for filtering + citation.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base",
    chunk_size=60,            # small for this toy doc; use 300-600 in prod
    chunk_overlap=10,
    separators=["\n\n", "\n", ". ", " ", ""],
)
docs = splitter.create_documents([DOC], metadatas=[{"source": "acme-10k",
                                                    "section": "full"}])
for i, d in enumerate(docs):
    print(f"[{i}] ({len(enc.encode(d.page_content))} tok) "
          f"{d.page_content[:70]!r}  meta={d.metadata}")

## 3. Structure-aware splitting

When the format is known, respect it. Markdown splits by headers (keeping the
header path as metadata); code splits on function/class boundaries so logic stays
intact.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, Language, RecursiveCharacterTextSplitter as RCTS

MD = """# Acme Report
## Risk Factors
Supply chain disruption cut costs by 40% after a new supplier contract.
## Financials
Revenue grew 12% to $4.2B; margin expanded 300bps."""

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2")])
for d in md_splitter.split_text(MD):
    print(d.metadata, "->", d.page_content[:60])

print("\n--- code-aware ---")
code_splitter = RCTS.from_language(language=Language.PYTHON,
                                   chunk_size=120, chunk_overlap=0)
CODE = "def a():\n    return 1\n\ndef b():\n    return 2\n"
for d in code_splitter.create_documents([CODE]):
    print(repr(d.page_content))

## 4. Semantic chunking **from scratch**

No framework — this *is* the algorithm:
1. Split into sentences
2. Embed each sentence
3. Cosine **distance** between consecutive sentences: $d_i = 1 - \cos(e_i, e_{i+1})$
4. Cut wherever $d_i$ exceeds the 95th percentile (a topic shift)

We provide a real OpenAI embedder and a **local fallback** so this runs offline.

In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.replace("\n", " "))
            if s.strip()]

def local_embed(texts, dim=256):
    "Deterministic hashing embedder so the notebook runs with no API key."
    vecs = []
    for t in texts:
        v = np.zeros(dim)
        for tok in re.findall(r"[a-z0-9]+", t.lower()):
            v[hash(tok) % dim] += 1.0
        n = np.linalg.norm(v)
        vecs.append(v / n if n else v)
    return np.array(vecs)

def embed(texts):
    key = os.getenv("OPENAI_API_KEY")
    if key:
        from openai import OpenAI
        client = OpenAI()
        r = client.embeddings.create(model="text-embedding-3-small", input=texts)
        return np.array([d.embedding for d in r.data])
    return local_embed(texts)

def cosine_distance(a, b):
    return 1.0 - float(np.dot(a, b) /
                       (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

In [ ]:
sentences = split_sentences(DOC)
emb = embed(sentences)
dists = np.array([cosine_distance(emb[i], emb[i+1]) for i in range(len(emb)-1)])
threshold = np.percentile(dists, 75)   # lower pct for a short doc

print("consecutive-sentence distances:")
for i, d in enumerate(dists):
    mark = "  <-- BREAK" if d > threshold else ""
    print(f"  s{i}->s{i+1}: {d:.3f}{mark}")

breaks = [i+1 for i, d in enumerate(dists) if d > threshold]
chunks, start = [], 0
for bp in breaks + [len(sentences)]:
    chunks.append(" ".join(sentences[start:bp])); start = bp
print(f"\n{len(chunks)} semantic chunks:")
for i, c in enumerate(chunks):
    print(f"  [{i}] {c[:80]}")

## 5. Contextual retrieval (Anthropic)

For each chunk, an LLM writes a 1–2 sentence context that situates it in the whole
document. You embed **context + chunk**, not the bare chunk. Anthropic reports
**−49%** failed retrievals (−67% with reranking).

Below is the exact prompt and the logic; it uses the LLM only if a key is present.

In [ ]:
CONTEXT_PROMPT = """<document>
{doc}
</document>
Here is a chunk we want to situate within the document:
<chunk>
{chunk}
</chunk>
Give a short (1-2 sentence) context situating this chunk within the overall
document so it can be understood standalone. Answer ONLY with the context."""

def contextualize(doc, chunk):
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        # offline stand-in so the cell runs; real runs call the model
        return "[context: from the Acme 2023 annual report]"
    from openai import OpenAI
    client = OpenAI()
    msg = CONTEXT_PROMPT.format(doc=doc, chunk=chunk)
    r = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        messages=[{"role": "user", "content": msg}])
    return r.choices[0].message.content.strip()

bare = "This cut costs by 40%."
enriched = f"{contextualize(DOC, bare)}\n\n{bare}"
print("BARE    :", bare)
print("ENRICHED:", enriched)
print("\nEmbed the ENRICHED text. Tip: prompt-cache the document prefix to keep cost down.")

## 6. Evaluation harness — pick a config with evidence

Build a labeled question set once, then sweep chunk configs and measure:
- **recall@k** — did any top-k chunk contain the answer substring?
- **MRR** — how highly did the first correct chunk rank?

Expect a non-monotonic curve: recall peaks at a middle size (tiny = fragmented,
huge = diluted).

In [ ]:
QA = [
    {"q": "How much did the supplier contract cut costs?", "a": "cut costs by 40%"},
    {"q": "How much did revenue grow?",                    "a": "grew 12%"},
    {"q": "How much was invested in R&D?",                 "a": "$520M in R&D"},
]

def build(text, c, o):
    sp = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="o200k_base", chunk_size=c, chunk_overlap=o)
    ch = [d.page_content for d in sp.create_documents([text])]
    return ch, embed(ch)

def retrieve(qv, cv, k):
    sims = cv @ qv / (np.linalg.norm(cv, axis=1) * np.linalg.norm(qv) + 1e-9)
    return np.argsort(-sims)[:k]

def evaluate(text, qa, configs, k=3):
    rows = []
    for c, o in configs:
        chunks, cv = build(text, c, o)
        qv = embed([x["q"] for x in qa])
        hits, rr = 0, []
        for x, v in zip(qa, qv):
            top = retrieve(v, cv, k)
            ranks = [r for r, idx in enumerate(top, 1)
                     if x["a"].lower() in chunks[idx].lower()]
            if ranks: hits += 1; rr.append(1/ranks[0])
            else: rr.append(0.0)
        rows.append({"size": c, "overlap": o, "n": len(chunks),
                     f"recall@{k}": round(hits/len(qa), 2),
                     "MRR": round(float(np.mean(rr)), 2)})
    return sorted(rows, key=lambda r: (-r[f"recall@{k}"], -r["MRR"]))

for row in evaluate(DOC, QA, [(40,0),(60,10),(100,15),(200,0)], k=3):
    print(row)

> With the local fallback embedder the numbers are illustrative. Set
`OPENAI_API_KEY` for meaningful semantic scores. The **harness itself** is the
reusable deliverable — it becomes your retrieval regression suite.

## 7. Exercises (prove you can implement it)

1. Change `chunk_size` in §2 and watch the chunk count match the §1 formula.
2. In §4, raise the percentile from 75 → 90. Do you get fewer, larger chunks? Why?
3. Add a `buffer` window to §4 (embed each sentence with its neighbors) and see if
   it removes any false breakpoints.
4. Add 5 more questions to `QA` in §6 and re-run the sweep. Where does recall peak?
5. Implement **late chunking**: embed the whole doc with a token-level model and
   mean-pool per chunk (see `04 Code Snippets/LLM/Late Chunking with Token Embeddings`).
6. Break the doc's table across a chunk boundary and observe retrieval degrade.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Chunking Strategies`
- Snippets: `04 Code Snippets/LLM/…`
- MOC: `06 Maps of Content/LLM Engineering Concepts`
